# Adım 1 — Docker Ortamının Kurulumu

Bu notebook, projenin Docker mimarisini ve tüm servislerin nasıl çalıştırıldığını açıklar.

---

## 1.1 Proje Mimarisi

Tüm bileşenler Docker konteynerleri içinde çalışır ve `docker-compose.yml` ile tek komutla ayağa kaldırılır.

```
┌──────────────────────────────────────────────────────────────┐
│                    Docker Compose Network                    │
│                                                              │
│  ┌─────────────┐   ┌─────────────┐   ┌──────────────────┐   │
│  │  Zookeeper   │──▶│    Kafka     │◀──│  Kafka Producer  │   │
│  │  :2181       │   │  :9092/29092 │   │  (Python)        │   │
│  └─────────────┘   └──────┬──────┘   └──────────────────┘   │
│                           │                                  │
│                    ┌──────▼──────┐                           │
│                    │   Kafka UI   │                           │
│                    │   :8080      │                           │
│                    └─────────────┘                           │
│                                                              │
│  ┌───────────────────────┐   ┌────────────────┐             │
│  │   Spark Workspace     │   │  MLflow Server  │             │
│  │   (Jupyter Lab)       │   │   :5001         │             │
│  │   :8888 / :4040       │   └────────────────┘             │
│  └───────────────────────┘                                  │
└──────────────────────────────────────────────────────────────┘
```

## 1.2 Servisler ve Portlar

| Servis | Port | Açıklama |
|--------|------|----------|
| Zookeeper | 2181 | Kafka cluster yönetimi |
| Kafka Broker | 9092 (host) / 29092 (container) | Mesaj kuyruğu sistemi |
| Kafka UI | 8080 | Topic ve mesaj izleme arayüzü |
| Spark Workspace (Jupyter Lab) | 8888 | Notebook çalışma ortamı |
| Spark UI | 4040 | Spark job izleme (Spark çalışınca aktif) |
| MLflow Server | 5001 | Model deney takibi arayüzü |
| Kafka Producer | - | Veri akışı servisi (arka planda çalışır) |

## 1.3 Docker Compose Dosyası

Projenin kök dizinindeki `docker-compose.yml` dosyası 6 servis tanımlar:

1. **Zookeeper** — Kafka'nın koordinasyon servisi
2. **Kafka** — Mesaj kuyruğu, streaming veri altyapısı
3. **Kafka UI** — Web tabanlı Kafka yönetim paneli
4. **Spark Workspace** — PySpark + Jupyter Lab (bu notebook'ların çalıştığı yer)
5. **Kafka Producer** — NYC Taxi verisini Kafka'ya stream eden Python uygulaması
6. **MLflow Server** — Makine öğrenmesi deney takibi

## 1.4 Çalıştırma Komutları

### Tüm servisleri başlatma:
```bash
docker-compose up -d --build
```

### Servislerin durumunu kontrol etme:
```bash
docker-compose ps
```

### Logları izleme:
```bash
# Tüm servisler
docker-compose logs -f

# Sadece kafka-producer
docker-compose logs -f kafka-producer
```

### Servisleri durdurma:
```bash
docker-compose down
```

## 1.5 Dockerfile'lar

### Spark Workspace (`spark-env/Dockerfile`)
```dockerfile
FROM jupyter/pyspark-notebook:spark-3.5.0
USER root
RUN apt-get update && apt-get install -y gcc
USER $NB_UID
RUN pip install kafka-python mlflow delta-spark pandas
```
- PySpark 3.5.0 yüklü Jupyter Notebook imajı kullanılır
- Kafka, MLflow ve Delta Lake Python kütüphaneleri eklenir

### Kafka Producer (`kafka-producer/Dockerfile`)
```dockerfile
FROM python:3.9-slim
WORKDIR /app
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt
COPY producer.py .
CMD ["python", "producer.py"]
```
- Hafif Python imajı kullanılır (Spark gerektirmez)
- Sadece `kafka-python`, `pandas`, `pyarrow` kurulur

## 1.6 Volume Mount'lar

| Host Yolu | Container Yolu | Açıklama |
|-----------|----------------|----------|
| `./notebooks` | `/home/jovyan/work/notebooks` | Jupyter notebook'ları |
| `./data` | `/home/jovyan/work/data` (Spark) | Veri dosyaları ve Delta Lake tabloları |
| `./data` | `/app/data` (Producer) | Producer'ın okuduğu parquet dosyası |
| `./mlruns` | `/home/jovyan/work/mlruns` (Spark) | MLflow deney logları |
| `./mlruns` | `/mlruns` (MLflow) | MLflow sunucu verisi |

## 1.7 Doğrulama

Tüm servislerin çalıştığını doğrulamak için aşağıdaki adresleri ziyaret edin:

- **Kafka UI**: [http://localhost:8080](http://localhost:8080)
- **Jupyter Lab**: [http://localhost:8888](http://localhost:8888)
- **MLflow**: [http://localhost:5001](http://localhost:5001)
- **Spark UI**: [http://localhost:4040](http://localhost:4040) *(Spark job çalışırken aktif)*